# Module 7, Section 3: Engine

Section 1 found the 11 wrong return-window answers with an online evaluator. Section 2 saw the same failure as an Insights subcategory with an error rate of 0%. This section follows the failure into LangSmith Engine, which turns it into an issue to fix.

**LangSmith Engine reads the traces of a tracing project on a schedule and files recurring problems as issues, each with a diagnosis, the traces that support it and a proposed fix.** The docs describe a loop of five steps:

1. Detect a recurring issue in the traces.
2. Diagnose the root cause against the traces and any connected source code.
3. Propose a fix, as a pull request when a code repository is connected.
4. Track the issue: add new traces that match it, and generate dataset examples that verify a fix.
5. Reopen the issue when it comes back after it was closed.

The docs separate Engine from Insights in two sentences: "Use Insights to understand how people use your agent. Use Engine to find out what is going wrong and what to change."

In this section:
1. What Engine reads, and whether the TechHub traces carry it
2. The issue board, and one issue checked against its evidence
3. Triage: the next action for each issue
4. A test case with two assertions, built from the evidence
5. The proposed fix, and what would prove it
6. The same steps in LangSmith

## Setup

This section calls no model and needs no key. Engine is not on every plan, so the core path reads a sample issue board that ships with the module. The board is hand-built from the seeded day and is not real Engine output. Its issues use the field names of the documented API.

Who can use Engine depends on how you reach LangSmith:

| Where you are | Turn Engine on for the organization | Start a scan of a project | Read issues, open evidence, add offline examples |
|---|---|---|---|
| Your own free organization (Developer plan) | No. Engine is not on the Developer plan, and personal organizations cannot use it | No | No. Use this notebook |
| A new workshop organization in which you are the Organization Admin | Yes | Yes. The first issues take up to 20 minutes | Yes, on your own issues once the scan has finished |
| A workspace in the facilitator's organization, with the default Editor role | No. An Organization Admin turns it on once for the whole organization | Yes, once the organization has Engine. Every attendee's scans count against one organization spend limit | Yes on your own project, or read-only on a board the facilitator prepared |
| No LangSmith key | No | No | This notebook, on the sample board |

Opening a pull request from an issue also needs a GitHub or GitLab repository connected to Engine. The facilitator shows that step.

If your organization has Engine and it has scanned your TechHub project, set `M7_ENGINE_ENABLED=true` in `.env`. The issue list of §2 then also reads your own issues. The exercises keep using the sample board, because its evidence traces are in the seed.

In [ ]:
import json, os, sys, warnings
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(REPO_ROOT))
# Load .env before any workshop import: config.py picks the models from it at import time
load_dotenv(REPO_ROOT / ".env")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=r".*is deprecated and will be removed after", category=DeprecationWarning)

from utils.langsmith_links import setup_langsmith
from utils.trace_seeding import describe, load_seed, print_trace
from utils.trace_upload import resolve_destination

LANGSMITH_AVAILABLE = setup_langsmith()            # one probe at most, one printed line
DEST = resolve_destination(LANGSMITH_AVAILABLE)    # project name, project id, navigation sentence
DATA = REPO_ROOT / "workshop_modules" / "module_7" / "data"
seed = load_seed()                                 # timestamps shifted so the newest run is 5 minutes old

## 1. What Engine reads

Engine looks for three kinds of trace: traces that raise errors, traces that behave unusually, and traces that look normal and hold a problem, such as a wrong answer that raises no error. Feedback is a signal too. Engine uses scores from online evaluators and annotation queues, ratings from your users, and traces that someone flagged for Engine. You set nothing up for this, and feedback added after a trace ran still counts.

Three things on a trace help Engine:

| On the trace | Why Engine uses it |
|---|---|
| `revision_id` in the metadata | It ties each trace to the commit that produced it, so Engine can compare the commits between two releases. The SDK reads it from the environment variable `LANGSMITH_REVISION_ID` |
| Feedback keys | A score such as `wrong_return_window = 1` points Engine at a failure that raises no error |
| A run name and a metadata value that every trace carries | They define which traces Engine scans |

You can narrow a scan to at most two conditions: one **Run Name** and one **Metadata** condition. You cannot scope a scan by feedback key, evaluator name or score threshold.

The next cell checks whether the seeded TechHub day carries these signals.

In [ ]:
from collections import Counter
from utils.trace_seeding import root_runs

roots = root_runs(seed)
runs = seed["runs"]
for name in ("revision_id", "environment", "app_version"):
    values = Counter(run["extra"]["metadata"].get(name) for run in runs if name in run["extra"]["metadata"])
    print(f"{name:<12} on {sum(values.values()):,} of {len(runs):,} runs. Values: {', '.join(str(value) for value in sorted(values))}")
commits = {run["extra"]["metadata"]["app_version"]: run["extra"]["metadata"]["revision_id"] for run in roots}
print("Release and commit: " + "; ".join(f"{version} is commit {commit}" for version, commit in sorted(commits.items())))
print()

feedback = Counter(row["key"] for row in seed["feedback"])
print("Feedback keys in the seed: " + ", ".join(f"{key} ({count} scores)" for key, count in feedback.items()))
print("Section 1 adds wrong_return_window, the online evaluator's score. Engine reads both with no setup.")
print()

root_names = Counter(run["name"] for run in roots)
environments = Counter(run["extra"]["metadata"]["environment"] for run in runs)
print("A trace scope for Engine that fits these traces:")
for name, count in root_names.items():
    print(f"  Run Name is {name}        (the root run of all {count} traces)")
for value, count in environments.items():
    print(f"  environment is {value}     (on {count:,} of {len(runs):,} runs)")

## 2. Read the issue board

Each issue on an Engine board has these fields in the API:

| Field | What it holds |
|---|---|
| `name`, `description` | A title, and the diagnosis in a few sentences |
| `severity` | `0` urgent, `1` high, `2` medium, `3` low. The UI calls it priority and offers Low, Medium and High |
| `status` | `open`, `fixing`, `watching`, `completed` or `ignored` |
| `type` | `actionable` (something can be fixed) or `finding` (an observation only). The default is `actionable` |
| `tags` | The failure category, such as `Silent tool error` or `Hallucination` |
| `proposed_fix` | The fix Engine proposes, as text. It can be missing |
| `fixes` | One entry per fix opened for the issue, newest first, each with a `branch` and a `pr_number` |

The SDK reads a board with `Client.list_project_issues(project_name)`, which returns a list of dictionaries. Its docstring lists keys such as `title`, `priority`, `category` and `trace_ids`. The current API has none of them, so read the fields of the table above, with `.get(...)`. The method also takes a `priority` argument, which the endpoint does not filter on: it returns every issue. This notebook filters and sorts in Python on `severity` and `status` instead.

In [ ]:
from utils.trace_seeding import SEVERITY_LABELS, load_engine_issues
from utils.trace_upload import new_client

ENGINE_ENABLED = os.getenv("M7_ENGINE_ENABLED") == "true"  # set it in .env when Engine has scanned your TechHub project
BOARD = load_engine_issues(seed)  # the sample board; its timestamps move with the seed

issues = None
if LANGSMITH_AVAILABLE and ENGINE_ENABLED:
    try:
        issues = new_client().list_project_issues(DEST.project_name)  # no priority argument: the endpoint does not filter on it
    except Exception as exc:  # noqa: BLE001 - any failure falls back to the sample board
        print(f"⏭️  Reading the issues failed ({type(exc).__name__}). Continuing with the sample board.")
    if issues == []:
        print("⏭️  Engine has filed no issue for this project yet. A first scan takes up to 20 minutes. Continuing with the sample board.")
        issues = None
    elif issues:
        print(f"Your board: {len(issues)} issues for project {DEST.project_name}. The exercises below use the sample board.")
if issues is None:
    reason = "LangSmith is not available" if not LANGSMITH_AVAILABLE else "M7_ENGINE_ENABLED is not true" if not ENGINE_ENABLED else "No issues from Engine"
    print(f"⏭️  {reason}: reading the sample issue board shipped with the module (hand-built from the seed, not real Engine output).")
    issues = BOARD["issues"]
print()


def issue_line(issue):
    severity = SEVERITY_LABELS.get(issue.get("severity"), "no severity")
    pull_requests = "".join(f"  PR {fix['pr_number']}" for fix in issue.get("fixes") or [] if fix.get("pr_number"))
    tags = ", ".join(issue.get("tags") or [])
    return f"[{severity}] {issue.get('status', 'no status'):<8} {issue.get('name', '(no name)')}  ({issue.get('type', 'actionable')}; {tags}){pull_requests}"


for issue in sorted(issues, key=lambda issue: issue.get("severity") if isinstance(issue.get("severity"), int) else 4):
    print(issue_line(issue))

### Check issue 1 against its evidence

Issue 1 says that the product-policy specialist answers without calling `search_policy_docs`. A reviewer checks a claim like this before acting on it. In LangSmith, **Open trace** on an evidence item opens the trace at the child run that caused the issue. Here the next cell prints the issue, then one evidence trace beside a trace from release 1.4.0 that answered the same kind of question correctly.

> 💡 **Try it first:** in the two trees, which tool run is in the correct trace and missing from the evidence trace? Under which run should it be?

In [ ]:
import textwrap
from utils.monitoring_checks import check
from utils.trace_seeding import load_labels, trace_runs

ISSUES = sorted(BOARD["issues"], key=lambda issue: issue["severity"])
issue_1 = ISSUES[0]


def show(text, indent="    "):
    print(textwrap.fill(text, width=100, initial_indent=indent, subsequent_indent=indent))


print(f"Issue 1: {issue_1['name']}")
print(f"  severity {issue_1['severity']} ({SEVERITY_LABELS[issue_1['severity']]}), status {issue_1['status']}, "
      f"type {issue_1['type']}, tags {', '.join(issue_1['tags'])}, {len(issue_1['fixes'])} fixes")
print("  Description")
show(issue_1["description"])
print("  Proposed fix")
show(issue_1["proposed_fix"])
print("  Evidence")
for item in BOARD["evidence"][issue_1["id"]]:
    show(f"{item['seed_trace_key']}: {item['comment']}")
print()

# The label file holds the ground truth of each conversation. It exists only because the data is
# synthetic. Here it finds a 1.4.0 conversation about an opened product, which got the right answer.
labels = load_labels()
evidence_key = BOARD["evidence"][issue_1["id"]][0]["seed_trace_key"]
correct_key = next(
    c["trace_keys"][0] for c in labels["conversations"]
    if c["topic"] == "return_window" and c["app_version"] == "1.4.0" and (c.get("details") or {}).get("item_condition") == "opened"
)
print("Evidence trace:")
print_trace(seed, evidence_key)
print()
print("A correct answer to the same kind of question:")
print_trace(seed, correct_key)
print()


def tools_by_specialist(key):
    """The tool runs inside each specialist run of one trace: {specialist name: [tool names]}."""
    members = trace_runs(seed, key)
    by_id = {run["id"]: run for run in members}
    found = {run["name"]: [] for run in members if by_id.get(run["parent_run_id"], {}).get("name") == "task"}
    for run in members:
        if run["run_type"] != "tool":
            continue
        parent = by_id.get(run["parent_run_id"])
        while parent is not None and parent["name"] not in found:
            parent = by_id.get(parent["parent_run_id"])
        if parent is not None:
            found[parent["name"]].append(run["name"])
    return found


correct_tools, evidence_tools = tools_by_specialist(correct_key), tools_by_specialist(evidence_key)
missing = [(agent, tool) for agent, tools in correct_tools.items() for tool in tools if tool not in evidence_tools.get(agent, [])]
for agent, tool in missing:
    print(f"{tool} runs under {agent} in {correct_key} and is missing from {evidence_key}.")
under_run, missing_tool = missing[0]
ok, message = check("s3_evidence", {"missing_tool": missing_tool, "under_run": under_run})
print(message)

## 3. Triage

Every issue needs a decision from a person. This notebook uses four action codes:

| Action code | In LangSmith | When |
|---|---|---|
| `apply_fix_and_add_examples` | **Open PR**, or copy the fix for a coding assistant, then **Add offline examples** | The issue is a real problem and nobody has started on it |
| `review_pull_request` | **View PR** | A fix is in flight: the status is `fixing` and a pull request exists |
| `save_for_later` | **Watch** in the docs pages, **Save for later** in the changelog | The issue is a finding: the answers are correct and there is nothing to fix now |
| `close_incorrectly_flagged` | **Close** > **Incorrectly Flagged** | A person read the evidence and the agent behaved correctly. No field of an issue says that, so a rule on fields never picks it |

Severity sets the order in which you work through the issues and plays no part in the choice of action.

Your decisions also teach Engine. On each scan, Engine turns the reviewer's recent actions into rules in the **User Preferences** section of its agent overview. Closing an issue or marking it **Incorrectly Flagged** lowers the priority of that category. Changing the priority records the severity you meant. Opening a pull request records that the category is worth fixing. Engine does not learn from automation, such as a pull request that merges, or from actions taken through LangSmith Chat.

The labels are moving. The docs pages say **Watch** and **Close**. The changelog renamed **Watch** to **Save for later** and replaced the single close button with a **Close** menu (**Done**, **Not Important**, **Incorrectly Flagged**). Use the label you see.

> 💡 **Try it first:** write `next_action(issue)` in `workshop_modules/module_7/exercises/triage.py`. It returns one of the four codes from the issue's fields. Check it from the repository root with `uv run python workshop_modules/module_7/exercises/check.py s3_triage`. The next cell holds the reference rule.

In [ ]:
from utils.monitoring_checks import TRIAGE_CASES


def next_action(issue):
    """The next action for one issue, decided from its type, status and fixes."""
    if issue.get("type") == "finding":
        return "save_for_later"  # an observation: the answers are correct
    if issue.get("status") == "fixing" and issue.get("fixes"):
        return "review_pull_request"  # a fix is already in flight
    return "apply_fix_and_add_examples"  # a real problem that nobody has started on


for number, board_issue in enumerate(ISSUES, start=1):
    fixes = board_issue.get("fixes") or []
    print(f"Issue {number}: type {board_issue['type']}, status {board_issue['status']}, {len(fixes)} in fixes: {next_action(board_issue)}")
print()
print("Two extra cases the checker also runs:")
for what, extra_issue, expected, why in TRIAGE_CASES:
    print(f"  {what}: {next_action(extra_issue)}")
print()
ok, message = check("s3_triage", next_action)
print(message)

## 4. Turn evidence into test cases

An issue's evidence can become test cases for a dataset. In LangSmith, **Add offline examples** at the top right of the **Evidence** list opens a dialog with the input, the wrong output the agent produced and a proposed expected output. **Add to Dataset** adds the examples directly. **Edit in annotation queue** lets a person review them first. In the queue, the reference output of each example is a list of named assertions. **An assertion is a short claim about what a correct answer should or should not include.** You edit them, add more with **+ Add assertion**, then click **Add to Dataset & Continue**. LangSmith attaches its Assertions evaluator when the examples are saved this way.

A useful assertion holds for the corrected answer and fails for the answer the agent recorded. An assertion that holds for both would not have caught the wrong answer. One that fails for the corrected answer would fail a correct agent.

> 💡 **Try it first:** the next cell prints the recorded answer, the corrected answer and six candidate assertions. Pick the two that hold for the corrected answer and fail for the recorded one, and write their keys in `MY_ASSERTIONS` before you run the cell.

In [ ]:
from utils.monitoring_checks import ASSERTION_CANDIDATES, assertion_case, assertion_table

case = assertion_case(seed, BOARD)  # the trace that the sample board proposes as an offline example for issue 1
print(f"Trace {case['trace_key']}")
print("Customer:          " + case["question"])
print("Recorded answer:   " + case["recorded_answer"])
print("Corrected answer:  " + case["corrected_answer"])
print()
print("Candidate assertions:")
for candidate in ASSERTION_CANDIDATES:
    print(f"  {candidate['key']:<28} {candidate['comment']}")
print()

MY_ASSERTIONS = None  # your two keys, for example ["key_one", "key_two"]. None lets the table below pick them
table = assertion_table(case)  # each candidate tried on both answers; a wording check stands in for a judge
picked = MY_ASSERTIONS if MY_ASSERTIONS is not None else [row["key"] for row in table if row["separates"]]
if MY_ASSERTIONS is None:
    print("MY_ASSERTIONS is None, so the cell picks the rows that hold for the corrected answer and fail for the recorded one.")
ok, message = check("s3_assertions", picked)
print(message)
print()
print(f"  {'assertion':<28} {'recorded':<10} {'corrected':<10}")
for row in table:
    print(f"  {row['key']:<28} {'holds' if row['holds_for_recorded'] else 'fails':<10} {'holds' if row['holds_for_corrected'] else 'fails':<10}")
print()

example = {
    "inputs": {"messages": [{"role": "user", "content": case["question"]}]},
    "outputs": {
        "messages": [{"role": "assistant", "content": case["corrected_answer"]}],
        "assertions": [{"key": c["key"], "comment": c["comment"]} for c in ASSERTION_CANDIDATES if c["key"] in picked],
    },
    "metadata": {"source": "module_7_section_3", "seed_trace_key": case["trace_key"], "synthetic": True},
}
print("The offline example:")
print(json.dumps(example, indent=2))
print()

DATASET_NAME = "techhub-deep-agent-regressions"
CREATE_DATASET = False  # set to True (with a key) to add the example to the dataset
if CREATE_DATASET and LANGSMITH_AVAILABLE and ok:
    try:
        client = new_client()
        if not client.has_dataset(dataset_name=DATASET_NAME):
            client.create_dataset(DATASET_NAME, description="TechHub deep agent: answers corrected by reviewers (Module 7).")
        # Re-running adds nothing, and neither does Section 2's example: both are matched on seed_trace_key.
        if list(client.list_examples(dataset_name=DATASET_NAME, metadata={"seed_trace_key": case["trace_key"]}, limit=1)):
            print(f"{DATASET_NAME} already has an example from trace {case['trace_key']}. Nothing was added.")
        else:
            client.create_examples(dataset_name=DATASET_NAME, examples=[example])
            print(f"Added the example to {DATASET_NAME}.")
    except Exception as exc:  # noqa: BLE001 - the example is printed above, so a failure loses nothing
        print(f"⏭️  Adding the example failed ({type(exc).__name__}). Add it in the UI with the steps of §6.")
else:
    print(f"⏭️  Not added. With a key and CREATE_DATASET = True, this cell adds the example to {DATASET_NAME}.")

## 5. The proposed fix, and what would prove it

Release 1.5.0 added one sentence to the product-policy specialist's instructions, and issue 1 proposes to remove it. The next cell prints both. Who applies the fix depends on where it lives:

| Where the fix lives | What Engine does |
|---|---|
| Code, in a connected GitHub or GitLab repository | **Open PR** opens a pull request, and the button becomes **View PR**. Code fixes as pull requests are for agents built with Deep Agents, LangChain or LangGraph |
| Instructions or skills in a connected Context Hub repository | Engine stores the change as a proposal. It changes nothing until a signed-in user with write access to that repository applies it |
| No repository connected | The proposed fix is text. You can copy the fix context into a coding assistant |

Engine never merges. Every pull request goes through your own review and branch protection rules.

Before a fix ships, prove it offline. That is the job of Workshop 8: the Harbor task `managed_agent/evals/opened-item-return-window` asks about an opened keyboard delivered 20 days ago and passes when the answer cites the 14-day window and says no. This notebook does not run the agent again.

After the fix ships, Engine keeps linking new traces to the issue on its later scans, and reopens a closed issue when the failure comes back. You cannot force a scan: there is no run-now action. The online evaluator of Section 1 gives you an answer sooner, and its scores are one of the signals Engine reads.

**120-minute path:** the next cell runs that evaluator over two small follow-up sets of 10 traces each, the same 10 questions answered before and after the fix.

In [ ]:
from evaluators import wrong_return_window_evaluator
from utils.monitoring_local import apply_evaluator, flagged_trace_keys
from utils.trace_seeding import PLANTED_INSTRUCTION, load_followups

print("Release 1.5.0 added this sentence to the product-policy specialist's instructions:")
show(PLANTED_INSTRUCTION)
print("Issue 1 proposes:")
show(issue_1["proposed_fix"])
print()

# Two synthetic sets made by the seed generator. Release 1.5.1 and its commit id are invented, like the others.
followups = load_followups(seed)
for label, title in (("before_fix", "Before the fix"), ("after_fix", "After the fix")):
    part = followups[label]
    part_roots = root_runs(part["runs"])
    flagged = flagged_trace_keys(apply_evaluator(wrong_return_window_evaluator, part_roots))
    print(f"{title}: app_version {part['app_version']} (commit {part['revision_id']}), {len(part_roots)} traces, "
          f"{len(flagged)} flagged wrong_return_window = 1" + (f": {', '.join(flagged)}" if flagged else ""))
print()
print("One question about an opened product, answered after the fix:")
print_trace(followups["after_fix"]["runs"], "fa-01")

## 6. In the product

If your organization has Engine, follow these steps on your own project. Otherwise the facilitator shows them on a prepared project. The labels come from the LangSmith docs and can differ a little in your workspace.

| Step | In the UI |
|---|---|
| 1. Open your TechHub traces | **Tracing** > agent **TechHub support** > **Production**; in a project-based workspace, **Tracing** > project `techhub-deep-agent-production`. Set the time range to at least the last 2 days |
| 2. Open Engine | **Engine** in the sidebar, or the **Engine** tab of the tracing project. An Organization Admin first turns on **Enable Engine** under **Settings** > **Engine** |
| 3. Pick the project | **+ Set up another project**, then your TechHub project under **Choose a project to analyze**. Connecting a code repository is optional |
| 4. Analysis level | **Reduced**, **Standard** (the default) or **Expanded** |
| 5. Scope | Under **Focus on specific traces**: `Run Name is deep_support_agent` and `environment is production`, the two conditions of §1 |
| 6. Start | **Start Analyzing**. Engine first writes an agent overview document; review it, then click **Accept & Continue** |
| 7. Read an issue | Open it. Read the diagnosis, **Feedback cited by Engine**, **Evidence** and **Proposed Fix**, then **Open trace** on an evidence item |
| 8. Make test cases | **Add offline examples**, then **Add to Dataset** or **Edit in annotation queue** |
| 9. Point Engine at a trace | On any trace, **Flag for Engine**, with a note. Engine reviews flagged traces on its next scan |

Facts to know before you turn it on:

- An Organization Admin enables Engine once for the organization. It is on the Plus and Enterprise plans.
- The first scan takes up to 20 minutes, and there is no action that starts a scan now.
- Engine bills in LangChain Standard Units (LSUs). One LSU costs 1 US dollar, and one run uses about 7 to 45 LSUs. An organization that has never set a limit is capped at 750 LSUs a month. Set a spend limit under **Spend limit** in the Engine settings, and use **Pause** to stop the scans of a project.
- On LangSmith Cloud, Engine uses LangChain-managed models. A self-hosted installation can use LangChain-managed models only on AWS in the US; with your own model providers it runs on any cloud.
- Trace content sent to Engine can include user messages, tool outputs and personal data. The docs recommend masking sensitive data in the client before it is traced, which is the first lab of Section 4.
- Whether Engine analyses traces uploaded with past timestamps, such as the seeded day, is UNVERIFIED.

### What you did in code, and where it is in LangSmith

| Step in this notebook | Same step in the UI |
|---|---|
| Counting `revision_id`, `environment` and the root run name | **Focus on specific traces** in the setup panel |
| `client.list_project_issues(DEST.project_name)`, or the sample board | The **Engine** page, with **Filter issues** and **Sort issues** |
| `print_trace(...)` on an evidence trace and a correct one | **Open trace** on an evidence item |
| `next_action(issue)` | **Open PR**, **View PR**, **Watch** (**Save for later**) or **Close** |
| `MY_ASSERTIONS` and the offline example | **Add offline examples** > **Edit in annotation queue** > **+ Add assertion** |
| `apply_evaluator(...)` on the follow-up sets | The online evaluator of Section 1, whose scores Engine reads |

## Recap

- Engine reads traces and their feedback, files recurring failures as issues with evidence and a proposed fix, and reopens an issue when the failure comes back. An Organization Admin turns it on, on a Plus or Enterprise plan.
- Issue 1's evidence holds up: in the 1.5.0 trace the product-policy specialist answers without calling `search_policy_docs`, and the correct 1.4.0 trace calls it.
- Triage reads `type`, `status` and `fixes`: issue 1 needs the fix and test cases, issue 2 has pull request 42 to review, and issue 3 is a finding to save for later. Severity only sets the order.
- The test case keeps the two assertions that separate the answers, `states_14_day_window` and `no_30_day_claim`. The Section 1 evaluator flags 4 of 10 follow-up traces before the fix and 0 after it.

**Next:** Section 4 finds the email address stored in one seeded trace, masks it before upload, and reads the LLM Gateway's policy blocks.